# Extract train and test sets 

In [ ]:
"""
Extract train and test tables from the overall cleaned table.

- Train and test contain distinct, non-overlapping locations (location_id);
  all years are kept for every sampled location.
- Year is rescaled (year_sc = year - 2002).
- Continuous variables are standardized (mean 0, sd 1) with parameters
  computed on the TRAIN set only; the TEST set uses the same parameters.
- All exported tables are ordered by location_id, then year.
"""

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# ---------- Setup ----------

ID_COL = "location_id"
YEAR_COL = "year"

N_TRAIN = 100_000   # number of locations in the train set
N_TEST = 100_000    # number of locations each test set
N_TEST_SETS = 3     # how many test sets to extract
RANDOM_SEED = 42

# continuous variables to standardize (adds a "<var>_sc" column for each)
SCALE_VARS = [
    "tmmn_mean", "tmmx_mean", "pr_sum",
    "veg_tmmn_mean", "veg_tmmx_mean", "veg_pr_sum",
    "vegetation_length", "elevation_mean",
    "pr_sum_cv", "veg_pr_sum_cv",
]

out_dir_tables = Path("/Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_allyears/")
big_table_path = out_dir_tables / "table_wgs84_2002-2023_clean.parquet"

tag = f"seed{RANDOM_SEED}"
train_path = out_dir_tables / f"table_wgs84_2002-2023_train_{N_TRAIN}px_{tag}.parquet"
params_path = out_dir_tables / f"scaling_params_train_{N_TRAIN}px_{tag}.csv"

In [3]:
# ---------- Load the combined table ----------

big_df = pd.read_parquet(big_table_path)
n_years = big_df[YEAR_COL].nunique()
print(f"Loaded {len(big_df)} rows, {big_df[ID_COL].nunique()} pixels, {n_years} years")

missing_vars = [v for v in SCALE_VARS if v not in big_df.columns]
if missing_vars:
    raise ValueError(f"Variables to scale not found in table: {missing_vars}")

Loaded 52667626 rows, 2393983 pixels, 22 years


In [4]:
# ---------- Sample distinct train / test locations ----------

unique_ids = big_df[ID_COL].drop_duplicates().reset_index(drop=True)

n_available = len(unique_ids)
n_needed = N_TRAIN + N_TEST_SETS * N_TEST
if n_needed > n_available:
    raise ValueError(
        f"Requested {N_TRAIN} train + {N_TEST_SETS} x {N_TEST} test = {n_needed} locations "
        f"but only {n_available} are available."
    )

# one random permutation of all locations: train = first N_TRAIN, then consecutive
# blocks of N_TEST for each test set (guarantees no overlap between any of the sets)
shuffled_ids = unique_ids.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
train_ids = shuffled_ids.iloc[:N_TRAIN]
test_ids_list = [
    shuffled_ids.iloc[N_TRAIN + i * N_TEST : N_TRAIN + (i + 1) * N_TEST]
    for i in range(N_TEST_SETS)
]

# no location may appear in more than one set
all_sampled = pd.concat([train_ids] + test_ids_list)
assert all_sampled.is_unique, "train/test sets overlap!"


def extract_set(ids):
    """All years for the given locations, ordered by location_id, then year."""
    return (
        big_df[big_df[ID_COL].isin(ids)]
        .sort_values([ID_COL, YEAR_COL])
        .reset_index(drop=True)
    )


train_df = extract_set(train_ids)
test_dfs = [extract_set(ids) for ids in test_ids_list]

print(f"Train:  {train_df[ID_COL].nunique()} locations, {len(train_df)} rows")
for i, d in enumerate(test_dfs, start=1):
    print(f"Test {i}: {d[ID_COL].nunique()} locations, {len(d)} rows")

Train:  100000 locations, 2200000 rows
Test 1: 100000 locations, 2200000 rows
Test 2: 100000 locations, 2200000 rows
Test 3: 100000 locations, 2200000 rows


In [7]:
# ---------- Scaling parameters (from the TRAIN set only) ----------

year_min = 2002 

# pandas .std() uses ddof=1 (n-1), same as R's scale()
params = pd.DataFrame({
    "mean": train_df[SCALE_VARS].mean(),
    "sd": train_df[SCALE_VARS].std(),
})

zero_sd = params.index[params["sd"] == 0].tolist()
if zero_sd:
    raise ValueError(f"Zero variance in train set for: {zero_sd} - cannot standardize")

print(f"\nYear rescaling: year_sc = year - {year_min}")

# save parameters so the scaling can be reproduced / reversed later
params_out = params.copy()
params_out.loc["year_min"] = [year_min, float("nan")]
print("Scaling parameters (train set):")
print(params_out)

params_out.to_csv(params_path, index_label="variable")
print(f"Saved scaling parameters to {params_path}")


Year rescaling: year_sc = year - 2002
Scaling parameters (train set):
                          mean          sd
tmmn_mean            -0.238016   11.334300
tmmx_mean            11.567666   13.321835
pr_sum              393.324738  175.710587
veg_tmmn_mean        10.890724    4.156130
veg_tmmx_mean        23.570707    5.950903
veg_pr_sum          219.993805  151.715271
vegetation_length   169.995529  111.134949
elevation_mean      598.350525  617.180420
pr_sum_cv            20.338511    9.138863
veg_pr_sum_cv        28.232832    9.118645
year_min           2002.000000         NaN
Saved scaling parameters to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_allyears/scaling_params_train_100000px_seed42.csv


In [8]:
# ---------- Apply scaling to train and all test sets ----------

# scale continuous variables to mean 0 / sd 1 (using TRAIN parameters)
# rescale year to start at 0 (year_sc = year - year_min)

def apply_scaling(df):
    df = df.copy()
    df["year_sc"] = df[YEAR_COL] - year_min
    for v in SCALE_VARS:
        df[f"{v}_sc"] = (df[v] - params.loc[v, "mean"]) / params.loc[v, "sd"]
    return df


train_df = apply_scaling(train_df)
test_dfs = [apply_scaling(d) for d in test_dfs]

# sanity check: train should be ~mean 0 / sd 1; test sets will be close but not exact
sc_cols = [f"{v}_sc" for v in SCALE_VARS]
check = pd.DataFrame({"train_mean": train_df[sc_cols].mean(), "train_sd": train_df[sc_cols].std()})
for i, d in enumerate(test_dfs, start=1):
    check[f"test{i}_mean"] = d[sc_cols].mean()
    check[f"test{i}_sd"] = d[sc_cols].std()
print("\nScaled variable check (train should be ~0 / ~1):")
print(check.round(3))


Scaled variable check (train should be ~0 / ~1):
                      train_mean  train_sd  test1_mean  test1_sd  test2_mean  \
tmmn_mean_sc                 0.0       1.0      -0.005     0.999       0.002   
tmmx_mean_sc                 0.0       1.0      -0.006     0.999       0.001   
pr_sum_sc                    0.0       1.0       0.001     0.999       0.002   
veg_tmmn_mean_sc             0.0       1.0      -0.004     0.999       0.003   
veg_tmmx_mean_sc             0.0       1.0      -0.004     1.000       0.000   
veg_pr_sum_sc                0.0       1.0      -0.004     0.998      -0.002   
vegetation_length_sc         0.0       1.0      -0.006     0.998      -0.001   
elevation_mean_sc           -0.0       1.0       0.003     1.004      -0.004   
pr_sum_cv_sc                 0.0       1.0      -0.001     1.003      -0.000   
veg_pr_sum_cv_sc             0.0       1.0       0.002     1.017       0.002   

                      test2_sd  test3_mean  test3_sd  
tmmn_mean_sc  

In [9]:
 # ---------- Save ----------

train_df.to_parquet(train_path, index=False)
print(f"\nSaved train table ({len(train_df)} rows) to {train_path}")

for i, d in enumerate(test_dfs, start=1):
    test_path = out_dir_tables / f"table_wgs84_2002-2023_test{i}_{N_TEST}px_{tag}.parquet"
    d.to_parquet(test_path, index=False)
    print(f"Saved test table {i} ({len(d)} rows) to {test_path}")


Saved train table (2200000 rows) to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_allyears/table_wgs84_2002-2023_train_100000px_seed42.parquet
Saved test table 1 (2200000 rows) to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_allyears/table_wgs84_2002-2023_test1_100000px_seed42.parquet
Saved test table 2 (2200000 rows) to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_allyears/table_wgs84_2002-2023_test2_100000px_seed42.parquet
Saved test table 3 (2200000 rows) to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_allyears/table_wgs84_2002-2023_test3_100000px_seed42.parquet
